# BundesLens · goal-model research
Reproduce a leakage-safe XGBoost and bivariate Poisson forecast. Run this notebook from the project root after installing `analysis/requirements.txt`.
Six pre-match features summarize attack, defense, and recent form. The score distribution shares a latent Poisson component of 0.08. The final 15% of each training window calibrates the outcome temperature.

In [ ]:
import sys
sys.path.insert(0, 'analysis')
from etl import read_matches
from train import features, train
import pandas as pd
history = [m for season in range(2015, 2024) for m in read_matches(season)]
pd.DataFrame(history).describe()

In [ ]:
model = train(history)
pd.Series(model['metrics'], index=['RPS', 'Log-loss', 'Multiclass Brier'])

## Model interpretation
For a one-split tree, the interventional SHAP contribution is its selected leaf value minus the training-cover-weighted mean leaf value. Summing across all trees produces additive contributions without a browser-side SHAP dependency.

In [ ]:
x, y = features(history)
sample = x[-1]
effects = [0.] * 6
for s in model['home']:
    effects[s['f']] += (s['l'] if sample[s['f']] < s['t'] else s['r']) - s['mean']
pd.Series(effects, index=['Home attack','Home defense','Away attack','Away defense','Home form','Away form'])